All three CSVs contain exactly the same records, including the number of copies of each record. Only row order differs. The exact, order-independent comparisons at the end have zero unmatched rows for every pair, across all columns, including `Sector` and `ID`.

Each file contains 1,876,617 rows, the same 740 nonmissing company IDs, 868 nonmissing security IDs, 1,133 ISINs, and dates from 1980-01-02 through 2026-09-08. Company names, ISINs, and sector labels are trimmed only for the distinct-value counts; the record comparisons preserve the original text.

Use one file for analysis. Concatenating the three files would produce 5,629,851 rows and triple summed volumes without adding information. A join is unnecessary. `ID` is unique in the first file and the full-record comparisons establish the same records in the other files; `(SecurityId, Date)` is not unique and also includes missing security IDs, so a naive join on those columns can multiply rows.

The earlier comparison tables count differences at the same **row positions**, not conflicting values for matched records. Their apparent differences disappear when row order is ignored.

The matching analyses are in [EDA.ipynb](EDA.ipynb), [EDA2.ipynb](EDA2.ipynb), and [EDA3.ipynb](EDA3.ipynb). The filename-to-dataset mapping is shown in each notebook's CSV-loading cell.


In [1]:
from itertools import combinations, zip_longest
from collections import Counter
import numpy as np
from pathlib import Path
import pandas as pd

# Read all values as text to detect exact differences, including missing fields.
data_dir = Path("../Data")
files = sorted(data_dir.glob("*.csv"))
assert len(files) == 3
columns = [pd.read_csv(path, sep=";", nrows=0).columns.tolist() for path in files]
assert columns[0] == columns[1] == columns[2]
pairs = list(combinations(range(3), 2))
difference_counts = {pair: pd.Series(0, index=columns[0], dtype="int64") for pair in pairs}
different_rows = dict.fromkeys(pairs, 0)
non_sector_id_differences = dict.fromkeys(pairs, 0)
examples = {pair: {} for pair in pairs}
full_variants = {pair: [Counter(), Counter()] for pair in pairs}
content_variants = {pair: [Counter(), Counter()] for pair in pairs}
content_columns = [c for c in columns[0] if c not in ("Sector", "ID")]
base_key_hashes = []
base_id_hashes = []
profiles = [{"rows": 0, "first_date": None, "last_date": None,
             "CompanyId": set(), "SecurityId": set(), "ISIN": set(), "Name": set(), "Sector": set()}
            for _ in files]
readers = [pd.read_csv(path, sep=";", dtype=str, keep_default_na=False, chunksize=20000)
           for path in files]

for chunks in zip_longest(*readers):
    assert all(chunk is not None for chunk in chunks), "Files have different row counts"
    assert len({len(chunk) for chunk in chunks}) == 1
    chunks = [chunk.reset_index(drop=True) for chunk in chunks]
    base_key_hashes.append(pd.util.hash_pandas_object(chunks[0][["SecurityId", "Date"]], index=False).to_numpy())
    base_id_hashes.append(pd.util.hash_pandas_object(chunks[0]["ID"], index=False).to_numpy())
    for profile, chunk in zip(profiles, chunks):
        profile["rows"] += len(chunk)
        first, last = chunk["Date"].min(), chunk["Date"].max()
        profile["first_date"] = min(profile["first_date"] or first, first)
        profile["last_date"] = max(profile["last_date"] or last, last)
        for column in ("CompanyId", "SecurityId", "ISIN", "Name", "Sector"):
            profile[column].update(chunk[column].str.strip().loc[lambda s: s.ne("")])
    for pair in pairs:
        left, right = (chunks[i] for i in pair)
        different = left.ne(right)
        difference_counts[pair] += different.sum()
        different_rows[pair] += int(different.any(axis=1).sum())
        full_mask = different.any(axis=1)
        content_mask = different[content_columns].any(axis=1)
        for side, frame in enumerate((left, right)):
            full_variants[pair][side].update(map(tuple, frame.loc[full_mask].to_numpy()))
            content_variants[pair][side].update(map(tuple, frame.loc[content_mask, content_columns].to_numpy()))
        non_sector_id_differences[pair] += int(different.drop(columns=["Sector", "ID"]).any(axis=1).sum())
        for column in different.columns[different.any()]:
            if column not in examples[pair]:
                row = different.index[different[column]][0]
                examples[pair][column] = {
                    "Date": left.at[row, "Date"], "CompanyId": left.at[row, "CompanyId"],
                    "Name": left.at[row, "Name"].strip(),
                    "left": left.at[row, column], "right": right.at[row, column],
                }

file_summary = pd.DataFrame([
    {"file": path.name, "rows": p["rows"], "first_date": p["first_date"], "last_date": p["last_date"],
     **{column: len(p[column]) for column in ("CompanyId", "SecurityId", "ISIN", "Name", "Sector")}}
    for path, p in zip(files, profiles)
]).set_index("file")
display(file_summary)


,rows,first_date,last_date,CompanyId,SecurityId,ISIN,Name,Sector
file,,,,,,,,
BMG9156K1018_10.09.2026_14.59.34.csv,1876617,1980-01-02,2026-09-08,740,868,1133,1444,16
NO0010063308_10.09.2026_15.03.41.csv,1876617,1980-01-02,2026-09-08,740,868,1133,1444,16
NO0010884794_10.09.2026_15.01.38.csv,1876617,1980-01-02,2026-09-08,740,868,1133,1444,16


In [2]:
for pair in pairs:
    a, b = pair
    print(f"\n{files[a].name} vs {files[b].name}")
    print(f"Row positions with any difference: {different_rows[pair]:,}")
    print(f"Row positions differing outside Sector and ID: {non_sector_id_differences[pair]:,}")
    display(difference_counts[pair].loc[lambda s: s.gt(0)].rename("differing_rows").to_frame())
    for column in ("CompanyId", "SecurityId", "ISIN", "Name", "Sector"):
        left_only = profiles[a][column] - profiles[b][column]
        right_only = profiles[b][column] - profiles[a][column]
        print(f"{column}: {len(left_only)} only in left; {len(right_only)} only in right")
    display(pd.DataFrame(examples[pair]).T)



BMG9156K1018_10.09.2026_14.59.34.csv vs NO0010063308_10.09.2026_15.03.41.csv
Row positions with any difference: 1,304
Row positions differing outside Sector and ID: 844


,differing_rows
Internal code,810
SecurityId,844
CompanyId,842
Symbol,824
ISIN,842
BestBidPrice,842
BestAskPrice,840
Open,840
High,834
Low,836


CompanyId: 0 only in left; 0 only in right
SecurityId: 0 only in left; 0 only in right
ISIN: 0 only in left; 0 only in right
Name: 0 only in left; 0 only in right
Sector: 0 only in left; 0 only in right


,Date,CompanyId,Name,left,right
Sector,2022-07-05,12348,ABL GROUP,Industrials,Energy
ID,2022-07-05,12348,ABL GROUP,1539082,1529042
SecurityId,2007-04-10,8249,APL,64537,58730
CompanyId,2007-04-10,8249,APL,8249,7916
Symbol,2007-04-10,8249,APL,APLC,APL
ISIN,2007-04-10,8249,APL,CY0100170915,NO0010255862
BestBidPrice,2007-04-10,8249,APL,"81,5",
BestAskPrice,2007-04-10,8249,APL,82,85
Open,2007-04-10,8249,APL,82,85
High,2007-04-10,8249,APL,82,85



BMG9156K1018_10.09.2026_14.59.34.csv vs NO0010884794_10.09.2026_15.01.38.csv
Row positions with any difference: 1,111
Row positions differing outside Sector and ID: 257


,differing_rows
Internal code,214
SecurityId,256
CompanyId,254
Symbol,218
ISIN,254
BestBidPrice,256
BestAskPrice,256
Open,254
High,256
Low,254


... (2 outputs left out)


NO0010063308_10.09.2026_15.03.41.csv vs NO0010884794_10.09.2026_15.01.38.csv
Row positions with any difference: 1,747
Row positions differing outside Sector and ID: 893


,differing_rows
Internal code,840
SecurityId,892
CompanyId,888
Symbol,858
ISIN,888
BestBidPrice,890
BestAskPrice,888
Open,886
High,882
Low,882


CompanyId: 0 only in left; 0 only in right
SecurityId: 0 only in left; 0 only in right
ISIN: 0 only in left; 0 only in right
Name: 0 only in left; 0 only in right
Sector: 0 only in left; 0 only in right


,Date,CompanyId,Name,left,right
Sector,2022-07-11,12348,ABL GROUP,Industrials,Energy
ID,2022-07-11,12348,ABL GROUP,1529680,1539785
SecurityId,2007-04-10,7916,APL,58730,64537
CompanyId,2007-04-10,7916,APL,7916,8249
Symbol,2007-04-10,7916,APL,APL,APLC
ISIN,2007-04-10,7916,APL,NO0010255862,CY0100170915
BestBidPrice,2007-04-10,7916,APL,,"81,5"
BestAskPrice,2007-04-10,7916,APL,85,82
Open,2007-04-10,7916,APL,85,82
High,2007-04-10,7916,APL,85,82


In [3]:
# Counters preserve duplicate multiplicities and compare exact text values.
# Matching rows at matching positions cancel automatically; compare the remainder.
for pair in pairs:
    a, b = pair
    print(f"\n{files[a].name} vs {files[b].name}")
    full_left, full_right = full_variants[pair]
    left, right = content_variants[pair]
    print("Unmatched full rows after ignoring row order:",
          sum((full_left - full_right).values()), sum((full_right - full_left).values()))
    print("Unmatched rows after ignoring row order, Sector, and ID:",
          sum((left - right).values()), sum((right - left).values()))

base_rows = profiles[0]["rows"]
print("\nFirst file: excess rows over unique (SecurityId, Date) hashes:",
      base_rows - len(np.unique(np.concatenate(base_key_hashes))))
print("First file: excess rows over unique ID hashes:",
      base_rows - len(np.unique(np.concatenate(base_id_hashes))))
print("Hash-based uniqueness counts are diagnostic; exact multiset comparisons above do not use hashes.")



BMG9156K1018_10.09.2026_14.59.34.csv vs NO0010063308_10.09.2026_15.03.41.csv
Unmatched full rows after ignoring row order: 0 0
Unmatched rows after ignoring row order, Sector, and ID: 0 0

BMG9156K1018_10.09.2026_14.59.34.csv vs NO0010884794_10.09.2026_15.01.38.csv
Unmatched full rows after ignoring row order: 0 0
Unmatched rows after ignoring row order, Sector, and ID: 0 0

NO0010063308_10.09.2026_15.03.41.csv vs NO0010884794_10.09.2026_15.01.38.csv
Unmatched full rows after ignoring row order: 0 0
Unmatched rows after ignoring row order, Sector, and ID: 0 0

First file: excess rows over unique (SecurityId, Date) hashes: 208187


First file: excess rows over unique ID hashes:

 0
Hash-based uniqueness counts are diagnostic; exact multiset comparisons above do not use hashes.
